# 02 - Grover: tìm kiếm trong $\sqrt{N}$ bước

Deutsch-Jozsa và BV đều là bài toán "được thiết kế" cho lợi thế lộ ra. Grover khác hẳn: nó giải
một bài toán **hoàn toàn tổng quát**.

> **Bài toán.** Có $N = 2^n$ khả năng. Một (hoặc vài) trong số đó là "nghiệm". Ta có oracle nói
> được "cái này có phải nghiệm không", nhưng không có bất kỳ cấu trúc nào để suy luận.
> Tìm nghiệm.

Cổ điển: không có cách nào hơn ngoài thử lần lượt - trung bình $N/2$ lần, xấu nhất $N$ lần.
Grover: khoảng $\frac{\pi}{4}\sqrt{N}$ lần. Với $N = 10^6$ thì là ~785 thay vì ~500.000.

Lợi thế đây là **bình phương** (quadratic), không phải hàm mũ - nhưng bù lại nó áp dụng cho
rất nhiều bài toán, vì "tìm trong danh sách không cấu trúc" là hình dạng chung của vô số bài toán.

In [1]:
import numpy as np

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit

print("qiskit:", qiskit.__version__)

sampler = SamplerV2()


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def do(qc, shots=1024, reg="c"):
    """Chạy mạch và trả về counts từ thanh ghi cổ điển tên `reg`."""
    ket_qua = sampler.run([qc], shots=shots).result()[0]
    return getattr(ket_qua.data, reg).get_counts()

qiskit: 2.5.2


## 1. Phase oracle: đánh dấu nghiệm bằng dấu âm

Lần này ta dùng thẳng dạng **phase oracle** (không cần ancilla nữa, vì bài trước đã hiểu
kickback - dạng này chỉ là kết quả cuối cùng của nó):

$$U_\omega\lvert x\rangle = \begin{cases} -\lvert x\rangle & x = \omega \ (\text{nghiệm})\\
\ \ \ \lvert x\rangle & x \neq \omega\end{cases}$$

Cách làm với nghiệm $\omega$ cụ thể: đặt `x` lên các qubit đang 0 trong $\omega$ (để biến $\omega$
thành chuỗi toàn 1), áp một cổng **multi-controlled Z** (`mcp(pi, ...)` - tức "chỉ lật dấu khi tất
cả qubit đều bằng 1"), rồi `x` trả lại.

In [ ]:
def phase_oracle(n, nghiem):
    """Lật dấu riêng trạng thái |nghiem>."""
    qc = QuantumCircuit(n, name=f"U_{nghiem}")
    for i, bit in enumerate(reversed(nghiem)):   # little-endian
        if bit == "0":
            qc.x(i)
    qc.mcp(np.pi, list(range(n - 1)), n - 1)     # Z có nhiều đường control
    for i, bit in enumerate(reversed(nghiem)):
        if bit == "0":
            qc.x(i)
    return qc


n, nghiem = 3, "101"
print(phase_oracle(n, nghiem).draw())

# Kiểm tra: đúng một trạng thái bị lật dấu, đúng là |101>
kt = QuantumCircuit(n)
kt.h(range(n))
kt.compose(phase_oracle(n, nghiem), inplace=True)
print("\nsau oracle (để ý duy nhất một số hạng âm):")
print(" ", ket(Statevector(kt)))

## 2. Diffuser: biến dấu âm thành amplitude lớn

Oracle đã đánh dấu nghiệm, nhưng **xác suất đo vẫn y nguyên**: $|-a|^2 = |a|^2$. Dấu âm một mình
vô dụng - cần một bước nữa để biến nó thành amplitude lớn hơn.

Bước đó gọi là **diffuser**, thực chất là phép "**lật quanh giá trị trung bình**"
(inversion about the mean): mỗi amplitude $a_k$ biến thành $2\bar{a} - a_k$ với $\bar{a}$ là trung bình.

Hiệu quả trực quan: số hạng âm nằm **dưới** trung bình rất nhiều, nên sau khi lật nó nhảy lên
**trên** trung bình - tức amplitude tăng. Các số hạng khác tụt xuống một chút.

Mạch của diffuser: $H^{\otimes n}$, rồi "lật dấu trạng thái $\lvert 0\ldots0\rangle$", rồi
$H^{\otimes n}$ - y nguyên cấu trúc của oracle nhưng với nghiệm là $\lvert 0\ldots0\rangle$.

In [ ]:
def diffuser(n):
    """Lật quanh giá trị trung bình."""
    qc = QuantumCircuit(n, name="diffuser")
    qc.h(range(n))
    qc.x(range(n))
    qc.mcp(np.pi, list(range(n - 1)), n - 1)
    qc.x(range(n))
    qc.h(range(n))
    return qc


print(diffuser(3).draw())

# Theo dõi amplitude qua từng bước của một vòng lặp
sv = QuantumCircuit(n)
sv.h(range(n))
print("\nban đầu (đều nhau):")
print(" ", ket(Statevector(sv)))

sv.compose(phase_oracle(n, nghiem), inplace=True)
print("\nsau oracle (chỉ đổi DẤU, xác suất chưa đổi):")
print(" ", ket(Statevector(sv)))
print("  xác suất của |101>:", round(float(Statevector(sv).probabilities_dict()["101"]), 3))

sv.compose(diffuser(n), inplace=True)
print("\nsau diffuser (amplitude của nghiệm tăng hẳn):")
print(" ", ket(Statevector(sv)))
print("  xác suất của |101>:", round(float(Statevector(sv).probabilities_dict()["101"]), 3))

## 3. Lặp lại: mỗi vòng quay thêm một góc $2\theta$

Một **vòng Grover** = oracle + diffuser. Về mặt hình học, trạng thái nằm trong một mặt phẳng 2 chiều
(càng "nghiệm" càng "không phải nghiệm"), và mỗi vòng **quay** nó thêm một góc cố định $2\theta$
về phía nghiệm, với

$$\sin\theta = \sqrt{\frac{M}{N}} \quad (M = \text{số nghiệm})$$

Sau $k$ vòng, xác suất đo được nghiệm là

$$P(k) = \sin^2\big((2k+1)\theta\big)$$

Đây là công thức quan trọng nhất của bài. Nó nói hai điều:

- Số vòng tối ưu là $k \approx \dfrac{\pi}{4}\sqrt{N/M}$ - từ đó ra $O(\sqrt N)$.
- **Lặp nhiều hơn không tốt hơn.** Quá điểm tối ưu, trạng thái quay vượt qua nghiệm và xác suất
  **tụt xuống**. Grover không phải kiểu "càng chạy lâu càng chính xác".

In [ ]:
def grover(n, nghiem, k):
    qc = QuantumCircuit(n)
    qc.h(range(n))                                   # superposition đều
    for _ in range(k):
        qc.compose(phase_oracle(n, nghiem), inplace=True)
        qc.compose(diffuser(n), inplace=True)
    return qc


theta = np.arcsin(np.sqrt(1 / 2 ** n))
print(f"n = {n}, N = {2 ** n}, M = 1, theta = {theta:.4f} rad\n")
print(f"{'k':>3} {'P(nghiệm) mô phỏng':>20} {'sin^2((2k+1)theta)':>20}")
for k in range(6):
    p = float(Statevector(grover(n, nghiem, k)).probabilities_dict()[nghiem])
    print(f"{k:>3} {p:>20.3f} {np.sin((2 * k + 1) * theta) ** 2:>20.3f}")

k_toi_uu = int(np.floor(np.pi / 4 * np.sqrt(2 ** n)))
print(f"\nk tối ưu = floor(pi/4 * sqrt(N)) = {k_toi_uu}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 3.4))
for n_ve, mau in [(3, "tab:blue"), (5, "tab:orange"), (8, "tab:green")]:
    th = np.arcsin(np.sqrt(1 / 2 ** n_ve))
    ks = np.arange(0, int(np.pi / 4 * np.sqrt(2 ** n_ve)) * 3 + 2)
    ax.plot(ks, np.sin((2 * ks + 1) * th) ** 2, "o-", ms=3, color=mau,
            label=f"n={n_ve} (N={2 ** n_ve}), k tối ưu={int(np.floor(np.pi / 4 * np.sqrt(2 ** n_ve)))}")
ax.axhline(1.0, ls=":", c="gray", lw=1)
ax.set_xlabel("số vòng Grover k")
ax.set_ylabel("P(đo được nghiệm)")
ax.set_title("Lặp quá số vòng tối ưu thì xác suất TỤT")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

## 4. Chạy thật

Lấy $k$ tối ưu rồi đo. Với $n=3$ ta được ~94%: không phải 100%, nên trên máy thật người ta
chạy vài lần rồi lấy kết quả xuất hiện nhiều nhất - và vì **kiểm tra** một ứng viên là việc
dễ (chỉ cần gọi $f$ một lần), sai thì thử lại, không tốn kém gì.

In [ ]:
qc = grover(n, nghiem, k_toi_uu)
qc.measure_all()
print(qc.draw(fold=100))

counts = do(qc, shots=2048, reg="meas")
print(f"\ncounts (k={k_toi_uu}):", counts)
tim_duoc = max(counts, key=counts.get)
print(f"-> đoán: {tim_duoc} (thật: {nghiem}) | tỷ lệ trúng: {counts[tim_duoc] / 2048:.1%}")
display(plot_histogram(counts))

## 5. Nhiều nghiệm

Với $M$ nghiệm, chỉ có $\theta$ đổi: $\sin\theta = \sqrt{M/N}$, nên số vòng tối ưu **giảm**:
$k \approx \frac{\pi}{4}\sqrt{N/M}$. Càng nhiều nghiệm càng dễ tìm - hợp lý.

Oracle nhiều nghiệm thì chỉ cần lật dấu từng nghiệm.

In [ ]:
def phase_oracle_nhieu(n, cac_nghiem):
    qc = QuantumCircuit(n, name="U_nhieu")
    for ng in cac_nghiem:
        qc.compose(phase_oracle(n, ng), inplace=True)
    return qc


def grover_nhieu(n, cac_nghiem, k):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    for _ in range(k):
        qc.compose(phase_oracle_nhieu(n, cac_nghiem), inplace=True)
        qc.compose(diffuser(n), inplace=True)
    return qc


cac_nghiem = ["101", "011"]
M, N = len(cac_nghiem), 2 ** n
th = np.arcsin(np.sqrt(M / N))
print(f"M = {M}, N = {N}, k tối ưu = {int(np.floor(np.pi / 4 * np.sqrt(N / M)))}\n")

for k in range(4):
    p = Statevector(grover_nhieu(n, cac_nghiem, k)).probabilities_dict()
    tong = sum(float(p[ng]) for ng in cac_nghiem)
    print(f"  k={k}: P(tìm được MỘT trong hai nghiệm) = {tong:.3f} "
          f"| lý thuyết {np.sin((2 * k + 1) * th) ** 2:.3f}")

g = grover_nhieu(n, cac_nghiem, 1)
g.measure_all()
print("\ncounts (k=1):", do(g, shots=2048, reg="meas"))

## Tóm tắt

| | Nội dung |
|---|---|
| Oracle | $U_\omega$ lật dấu các nghiệm (`mcp(pi, ...)` + `x` hai bên) |
| Diffuser | lật quanh trung bình: $H^{\otimes n}$, lật dấu $\lvert 0\rangle$, $H^{\otimes n}$ |
| Một vòng | oracle + diffuser = quay góc $2\theta$ về phía nghiệm |
| Xác suất | $P(k) = \sin^2((2k+1)\theta)$, $\sin\theta=\sqrt{M/N}$ |
| Số vòng tối ưu | $k \approx \frac{\pi}{4}\sqrt{N/M}$ |
| Lợi thế | bình phương ($\sqrt N$), **không** phải hàm mũ |
| Bẫy | lặp quá số vòng tối ưu thì xác suất tụt xuống |

### Thử tự làm

1. Đổi `nghiem` thành `"000"` và chạy lại. Oracle lúc đó rút gọn còn mạch gì?
2. Chạy $n = 4$ với 1 nghiệm: tính $k$ tối ưu, đo thử với $k-1$, $k$, $k+1$ và đối chiếu công thức.
3. Thử 4 nghiệm với $n = 3$ (tức $M/N = 1/2$). Tính $\theta$, và giải thích vì sao lúc này
   Grover gần như không giúp gì.

Bài tiếp theo: `03_teleportation_va_superdense.ipynb`.